# 03 Primary-Key and Foreign-Key Checks

Checks natural keys and basic session-level referential integrity. Telemetry duplicate checks are skipped unless event-level keys exist.

In [1]:
from pathlib import Path
import sys
from datetime import datetime
import json

import numpy as np
import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

SHARED = ROOT / "eda" / "shared" / "scripts"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from config import (
    RAW_DATA_PATH,
    EXPECTED_ENDPOINTS,
    TELEMETRY_ENDPOINTS,
    CRITICAL_COLUMNS,
    PRIMARY_KEYS,
    FOREIGN_KEYS,
    RANGE_RULES,
    THRESHOLDS,
    TECHNICAL_KEY_COLUMNS,
    DOMAIN_REVIEW_COLUMNS,
    STRUCTURAL_OPTIONAL_COLUMNS,
    ALLOWED_NULL_SCENARIOS,
    VALIDATION_SEVERITY,
    RANGE_SEVERITY_OVERRIDES,
)
from file_utils import build_file_inventory, endpoint_files, endpoint_files, iter_csv_endpoint
from validation_utils import endpoint_columns, null_profile, duplicate_count, range_violations

NOTEBOOK_NAME = "03_pk_fk_checks"
OUTPUT_TABLES = ROOT / "eda" / "bronze" / "outputs" / "tables" / NOTEBOOK_NAME
OUTPUT_CHARTS = ROOT / "eda" / "bronze" / "outputs" / "charts" / NOTEBOOK_NAME
OUTPUT_REPORTS = ROOT / "eda" / "bronze" / "outputs" / "reports" / NOTEBOOK_NAME
INSIGHTS = ROOT / "eda" / "bronze" / "insights"
CHECKPOINTS = ROOT / "eda" / "bronze" / "checkpoints"
for path in [OUTPUT_TABLES, OUTPUT_CHARTS, OUTPUT_REPORTS, INSIGHTS, CHECKPOINTS]:
    path.mkdir(parents=True, exist_ok=True)

def write_report(name: str, payload: dict) -> None:
    (OUTPUT_REPORTS / f"{name}.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")

def write_insight(filename: str, title: str, summary: str, observations: list[str], issues: list[str], recommendations: list[str], next_steps: list[str]) -> None:
    content = f"# {title}\n\n"
    content += f"**Generated at:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += f"## Summary\n\n{summary}\n\n"
    content += "## Key Observations\n\n" + "\n".join(f"- {item}" for item in observations) + "\n\n"
    content += "## Issues\n\n" + ("\n".join(f"- {item}" for item in issues) if issues else "- None") + "\n\n"
    content += "## Recommendations\n\n" + "\n".join(f"- {item}" for item in recommendations) + "\n\n"
    content += "## Next Steps\n\n" + "\n".join(f"- {item}" for item in next_steps) + "\n"
    (INSIGHTS / filename).write_text(content, encoding="utf-8")

def p0_status_from_severity(df: pd.DataFrame) -> str:
    if df.empty or "severity" not in df.columns:
        return "PASS"
    blockers = df[df["severity"].eq("BLOCKER")]
    return "FAIL" if not blockers.empty else "PASS"

print("=" * 72)
print(f"BRONZE VALIDATION - {NOTEBOOK_NAME}")
print(f"Start time: {datetime.now()}")
print(f"Raw data path: {RAW_DATA_PATH}")
print("=" * 72)


BRONZE VALIDATION - 03_pk_fk_checks
Start time: 2026-06-01 17:59:26.749967
Raw data path: D:\F1_WinRate_Predictor\data\raw


In [2]:
records = []
chunk_size = int(THRESHOLDS.get("validation_chunk_size", 200000))
for endpoint, key in PRIMARY_KEYS.items():
    if endpoint in TELEMETRY_ENDPOINTS:
        records.append({"check_type": "pk", "endpoint": endpoint, "key": "|".join(key), "rows_checked": 0, "violations": 0, "status": "SKIP"})
        continue
    rows, duplicates = duplicate_count(RAW_DATA_PATH, endpoint, key, chunk_size)
    records.append({"check_type": "pk", "endpoint": endpoint, "key": "|".join(key), "rows_checked": rows, "violations": duplicates, "status": "PASS" if duplicates == 0 else "FAIL"})

parent_cache = {}
for fk in FOREIGN_KEYS:
    parent = fk["parent"]
    parent_key = fk["parent_key"]
    if parent not in parent_cache:
        values = set()
        for chunk in iter_csv_endpoint(RAW_DATA_PATH, parent, columns=[parent_key], chunksize=chunk_size):
            if parent_key in chunk.columns:
                values.update(chunk[parent_key].dropna().astype(str).tolist())
        parent_cache[parent] = values
    child_values = set()
    for chunk in iter_csv_endpoint(RAW_DATA_PATH, fk["child"], columns=[fk["child_key"]], chunksize=chunk_size):
        if fk["child_key"] in chunk.columns:
            child_values.update(chunk[fk["child_key"]].dropna().astype(str).tolist())
    orphans = child_values - parent_cache[parent]
    records.append({"check_type": "fk", "endpoint": fk["child"], "key": f"{fk['child_key']}->{parent}.{parent_key}", "rows_checked": len(child_values), "violations": len(orphans), "status": "PASS" if not orphans else "FAIL"})

pkfk_df = pd.DataFrame(records)
pkfk_df.to_csv(OUTPUT_TABLES / "pk_fk_validation.csv", index=False)
display(pkfk_df)

,check_type,endpoint,key,rows_checked,violations,status
0,pk,meetings,meeting_key,76,0,PASS
1,pk,sessions,session_key,140,0,PASS
2,pk,drivers,session_key|driver_number,2837,0,PASS
3,pk,laps,session_key|driver_number|lap_number,84072,0,PASS
4,pk,session_result,session_key|driver_number,2740,0,PASS
5,pk,starting_grid,session_key|driver_number,1355,0,PASS
6,pk,stints,session_key|driver_number|stint_number,9192,0,PASS
7,pk,weather,session_key|date,14902,0,PASS
8,pk,overtakes,session_key|date|overtaking_driver_number|over...,13995,0,PASS
9,pk,position,session_key|driver_number|date,115021,0,PASS


In [3]:
fig = px.bar(pkfk_df, x="endpoint", y="violations", color="check_type", barmode="group", title="PK/FK Validation Violations")
fig.update_xaxes(tickangle=35)
fig.write_html(OUTPUT_CHARTS / "pk_fk_violations.html", include_plotlyjs="cdn")
try:
    fig.write_image(OUTPUT_CHARTS / "pk_fk_violations.png")
except Exception:
    pass
fig.show()

In [4]:
failed = pkfk_df[pkfk_df["status"] == "FAIL"]
report = {"notebook": NOTEBOOK_NAME, "timestamp": datetime.now().isoformat(), "p0_status": "PASS" if failed.empty else "FAIL", "results": pkfk_df.to_dict("records")}
write_report("pk_fk_validation", report)
write_insight(
    "03_pk_fk_insights.md",
    "PK/FK Validation Insights",
    f"Executed {len(pkfk_df)} key-integrity checks.",
    [f"Failed checks: {len(failed)}"],
    [f"{row.check_type.upper()} {row.endpoint} {row.key}: {row.violations} violations" for row in failed.itertuples()],
    ["Investigate duplicate keys and orphan session references before Silver cleaning."],
    ["Run 04_null_analysis.ipynb"],
)
print(report["p0_status"])

PASS
